# Sierra_Leone — EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scripts.eda_utils import zscore_flags, impute_median

# Settings
pd.set_option("display.max_columns", 100)
plt.rcParams.update({"figure.figsize": (10, 5)})

# File paths (update if needed)
RAW_PATH = "data/sierra_leone.csv"   # put your raw file here
CLEAN_PATH = "data/sierra_leone_clean.csv"

In [ ]:
# Load data
df = pd.read_csv(RAW_PATH, parse_dates=["Timestamp"])
df.sort_values("Timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

## 1) Summary stats & missing values

In [ ]:
# Summary stats
display(df.describe(include='all'))

# Missing values report
na_counts = df.isna().sum().sort_values(ascending=False)
na_pct = (na_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing": na_counts, "pct": na_pct})
display(missing_report)

## 2) Outlier detection & basic cleaning

In [ ]:
numeric_cols = ["GHI","DNI","DHI","ModA","ModB","WS","WSgust"]
df_z = zscore_flags(df, numeric_cols, thresh=3.0)
# Flag counts
flags = {c: int(df_z[f"{c}_outlier"].sum()) for c in numeric_cols if f"{c}_outlier" in df_z.columns}
flags

In [ ]:
# Impute median for key columns
key_cols = ["GHI","DNI","DHI","ModA","ModB","Tamb","RH","WS","WSgust","BP","TModA","TModB"]
df_clean = impute_median(df, [c for c in key_cols if c in df.columns])
df_clean.to_csv(CLEAN_PATH, index=False)
print(f"Saved: {CLEAN_PATH}")

## 3) Time series analysis

In [ ]:
for col in ["GHI","DNI","DHI","Tamb"]:
    if col in df.columns:
        plt.figure()
        plt.plot(df["Timestamp"], df[col])
        plt.title(col)
        plt.xlabel("Timestamp")
        plt.ylabel(col)
        plt.show()

# Monthly patterns
df["month"] = df["Timestamp"].dt.to_period("M")
monthly = df.groupby("month")[["GHI","DNI","DHI","Tamb"]].mean(numeric_only=True)
display(monthly.tail())

## 4) Cleaning impact (ModA/ModB)

In [ ]:
if "Cleaning" in df.columns:
    grp = df.groupby("Cleaning")[["ModA","ModB"]].mean(numeric_only=True)
    display(grp)
    for c in ["ModA","ModB"]:
        if c in df.columns:
            plt.figure()
            grp_plot = grp[c]
            grp_plot.plot(kind="bar")
            plt.title(f"Average {c} by Cleaning flag")
            plt.xlabel("Cleaning flag")
            plt.ylabel(c)
            plt.show()

## 5) Correlation & relationships

In [ ]:
corr_cols = [c for c in ["GHI","DNI","DHI","TModA","TModB","Tamb","RH","WS","WSgust"] if c in df.columns]
if corr_cols:
    corr_mat = df[corr_cols].corr(numeric_only=True)
    display(corr_mat)

    # Simple heatmap with matplotlib
    plt.figure()
    plt.imshow(corr_mat, interpolation='nearest')
    plt.xticks(range(len(corr_cols)), corr_cols, rotation=45, ha='right')
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.colorbar()
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

# Scatter plots
pairs = [("WS","GHI"),("WSgust","GHI"),("WD","GHI"),("RH","Tamb"),("RH","GHI")]
for x, y in pairs:
    if x in df.columns and y in df.columns:
        plt.figure()
        plt.scatter(df[x], df[y], alpha=0.3)
        plt.xlabel(x); plt.ylabel(y); plt.title(f"{x} vs {y}")
        plt.show()

## 6) Wind & distributions

In [ ]:
# Histograms
for col in ["GHI","WS"]:
    if col in df.columns:
        plt.figure()
        plt.hist(df[col].dropna(), bins=30)
        plt.title(f"Histogram of {col}")
        plt.xlabel(col); plt.ylabel("Count")
        plt.show()

# Wind rose (requires 'windrose' package)
try:
    from windrose import WindroseAxes
    if "WS" in df.columns and "WD" in df.columns:
        ax = WindroseAxes.from_ax()
        ax.bar(df["WD"].values, df["WS"].values, normed=True, opening=0.8, edgecolor='white')
        ax.set_legend()
        plt.show()
except Exception as e:
    print("Windrose not available or failed:", e)

## 7) Bubble chart

In [ ]:
x, y, size = "GHI", "Tamb", "RH" if "RH" in df.columns else ("GHI","Tamb","BP")
if all(c in df.columns for c in [x, y, size]):
    plt.figure()
    plt.scatter(df[x], df[y], s=(df[size].fillna(0) + 1), alpha=0.3)
    plt.xlabel(x); plt.ylabel(y); plt.title(f"{x} vs {y} (size={size})")
    plt.show()